<a href="https://colab.research.google.com/github/Jayku88/22AIE301_Probabilistic_Reasoning/blob/main/22AIE301_Lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 22AIE301 - Probabilistic Reasoning
## Lab 04 - Graph Theory Primer

| | |
|---|---|
| **Name** | _________________________________ |
| **Roll No** | _____________ |
| **Date** | _____________ |
| **Lab Slot** | _____________ |

---

**Based on:** Lecture 4 - Graph Theory Primer

**Learning outcomes.** By the end of this lab, you will be able to:
- Build directed and undirected graphs in `networkx` and compute neighbours, degree, parents, children, in-degree, and out-degree.
- Distinguish a subgraph from an induced subgraph in code.
- Classify a sequence of nodes as a walk, trail, or path.
- Check connectivity and detect cycles in undirected and directed graphs.
- Implement Kahn's algorithm for topological ordering from scratch.
- Compute the skeleton of a DAG and determine whether it is a polytree.
- Compute ancestors, descendants, and non-descendants of a node, and verify the Local Markov Condition.
- Compute the Markov blanket of a node and detect v-structures.

**Instructions**
- Code cells with `___` are incomplete - replace each `___` with the correct code.
- Every task has an `assert` check right after it. If your code is correct, the cell runs silently; if not, you'll see an `AssertionError`.
- Section 12 is a capstone: you'll run every function you built on a DAG that is uniquely generated from **your own roll number** - so your capstone answers will differ from your classmates.
- Run cells top to bottom. Do not skip the Setup cell.
- Show your outputs to the facilitator before leaving the lab.



## Setup: Run this cell first before anything else

Run the cell below to import the libraries used throughout this lab.


In [ ]:
import random
import itertools

import networkx as nx
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (5, 4)


## Section 0 - Roll Number Seeding

Fill in your roll number below (numeric part only, e.g. if your roll number is `AM.EN.U4CSE21042`, use `42`).
This seeds a **personalized DAG** that you will use throughout Section 12.


In [ ]:
ROLL_NUMBER = ___  # TODO: enter the numeric part of your roll number as an int

assert isinstance(ROLL_NUMBER, int), "ROLL_NUMBER must be an integer"
SEED = ROLL_NUMBER
print(f"Using seed: {SEED}")

In [ ]:
def generate_personal_dag(seed, n_nodes=6, edge_prob=0.35):
    '''Generates a random DAG with `n_nodes` nodes, deterministic given `seed`.
    Edges only go from a lower-indexed node to a higher-indexed node,
    which guarantees the result is always acyclic.'''
    rng = random.Random(seed)
    nodes = [f"X{i}" for i in range(n_nodes)]
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    for i in range(n_nodes):
        for j in range(i + 1, n_nodes):
            if rng.random() < edge_prob:
                G.add_edge(nodes[i], nodes[j])
    return G

my_dag = generate_personal_dag(SEED)
print("Your personalized DAG edges:", list(my_dag.edges()))

fig, ax = plt.subplots()
pos = nx.spring_layout(my_dag, seed=SEED)
nx.draw(my_dag, pos, ax=ax, with_labels=True, node_color="lightblue",
        node_size=800, arrowsize=15, font_weight="bold")
ax.set_title(f"Your personalized DAG (seed={SEED})")
plt.show()


## Section 1- Graphs: Nodes and Edges

Recall: a graph $G=(V,E)$ consists of a node set $V$ and an edge set $E$. In `networkx`,
`nx.Graph()` is undirected and `nx.DiGraph()` is directed.

We will use the **Student BN** from lecture as our running example throughout this lab:
$D\to G,\; I\to G,\; I\to S,\; G\to L$.


In [ ]:
# TASK 1: Build the Student BN as a directed graph.
# Add the four edges: D->G, I->G, I->S, G->L

student_bn = nx.DiGraph()
student_bn.add_nodes_from(["D", "I", "G", "S", "L"])

student_bn.add_edge(___, ___)   # D -> G
student_bn.add_edge(___, ___)   # I -> G
student_bn.add_edge(___, ___)   # I -> S
student_bn.add_edge(___, ___)   # G -> L

assert set(student_bn.edges()) == {("D", "G"), ("I", "G"), ("I", "S"), ("G", "L")}
print("Student BN built correctly:", list(student_bn.edges()))

In [ ]:
fig, ax = plt.subplots()
pos = {"D": (0, 1), "I": (2, 1), "G": (1, 0), "S": (3, 0), "L": (1, -1)}
nx.draw(student_bn, pos, ax=ax, with_labels=True, node_color="lightyellow",
        node_size=900, arrowsize=15, font_weight="bold")
ax.set_title("Student BN")
plt.show()


## Section 2 — Neighbours, Degree, Parents, Children

For a directed graph:
- $\text{Pa}(X) = \{Y : (Y\to X)\in E\}$ — use `G.predecessors(X)`
- $\text{Ch}(X) = \{Y : (X\to Y)\in E\}$ — use `G.successors(X)`
- in-degree $= |\text{Pa}(X)|$, out-degree $=|\text{Ch}(X)|$


In [ ]:
# TASK 2: Complete this function using predecessors/successors.

def get_parents_children(G, node):
    parents = set(G.___(node))     # fill in: predecessors or successors?
    children = set(G.___(node))    # fill in: predecessors or successors?
    return parents, children

# Quick check on the Student BN
for node in ["D", "I", "G", "S", "L"]:
    pa, ch = get_parents_children(student_bn, node)
    print(f"{node}: Pa={pa}, Ch={ch}, in-deg={len(pa)}, out-deg={len(ch)}")

assert get_parents_children(student_bn, "G") == ({"D", "I"}, {"L"})
assert get_parents_children(student_bn, "S") == ({"I"}, set())
print("Section 2 checks passed.")


**Task 2b.** Now apply `get_parents_children` to *your* personalized DAG (`my_dag`) below,
for every node. No blanks here - just run it and read the output.


In [ ]:
for node in my_dag.nodes():
    pa, ch = get_parents_children(my_dag, node)
    print(f"{node}: Pa={pa}, Ch={ch}, in-deg={len(pa)}, out-deg={len(ch)}")


## Section 3 - Subgraphs vs. Induced Subgraphs

Recall the key distinction:
- **Induced subgraph** on $V'$: keeps *every* $G$-edge with both endpoints in $V'$. `networkx`'s
  `G.subgraph(V')` always gives you the **induced** subgraph - you don't get to choose the edges.
- **(General) subgraph** on $V'$: you choose *both* $V'$ and a subset of the eligible edges.

We'll use the undirected graph from lecture: edges $A{-}B,\,A{-}C,\,B{-}D,\,C{-}D,\,B{-}C$,
with $V'=\{A,B,C\}$.


In [ ]:
base_graph = nx.Graph()
base_graph.add_edges_from([("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("B", "C")])

V_prime = {"A", "B", "C"}

# TASK 3a: Get the INDUCED subgraph on V_prime using networkx.
induced = base_graph.___(V_prime)   # fill in the correct method name

assert set(induced.edges()) == {("A", "B"), ("A", "C"), ("B", "C")}
print("Induced subgraph edges:", list(induced.edges()))


In [ ]:
# TASK 3b: Build a valid NON-induced subgraph on V_prime that keeps only
# the edge A-B (drop A-C and B-C, even though both endpoints are in V_prime).

non_induced = nx.Graph()
non_induced.add_nodes_from(V_prime)
non_induced.add_edge(___, ___)   # add only the A-B edge

assert set(non_induced.edges()) == {("A", "B")}
assert set(non_induced.edges()) != set(induced.edges()), \
    "This should NOT equal the induced subgraph's edge set"
print("Non-induced subgraph edges:", list(non_induced.edges()))
print("Confirmed: every induced subgraph is a subgraph, but not every subgraph is induced.")


## Section 4 - Walks, Trails, and Paths

- **Walk**: consecutive nodes must be adjacent; nodes and edges may repeat.
- **Trail**: a walk with no *repeated edge*.
- **Path**: a walk with no *repeated node* (this also forbids repeated edges).

We'll use the lecture's example graph: edges $A{-}B,\,B{-}C,\,C{-}D,\,D{-}B,\,B{-}E$.


In [ ]:
wtp_graph = nx.Graph()
wtp_graph.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("D", "B"), ("B", "E")])

def classify_sequence(G, seq):
    '''Classify a node sequence as 'not a walk', 'walk', 'trail', or 'path'.'''
    edges_used = []
    for i in range(len(seq) - 1):
        u, v = seq[i], seq[i + 1]
        if not G.has_edge(u, v):
            return "not a walk"
        # store edge in a direction-independent (sorted) form
        edges_used.append(tuple(sorted((u, v))))

    is_trail = len(edges_used) == len(set(edges_used))
    is_path = len(seq) == len(set(seq))

    # TASK 4: Fill in the correct classification logic.
    if is_path:
        return ___          # a path is also a trail and a walk -- what single label applies?
    elif is_trail:
        return ___          # repeats a node but not an edge
    else:
        return ___          # repeats an edge (and therefore possibly a node too)

# Quick checks against the lecture's three examples
assert classify_sequence(wtp_graph, ["A", "B", "C", "B", "A"]) == "walk"
assert classify_sequence(wtp_graph, ["A", "B", "C", "D", "B", "E"]) == "trail"
assert classify_sequence(wtp_graph, ["A", "B", "C", "D"]) == "path"
print("Section 4 checks passed.")


## Section 5 - Connectivity and Cycles

- A graph is **connected** if there is a path between every pair of nodes.
- A **cycle** is a closed walk (or trail) that starts and ends at the same node, using no
  repeated edges.


In [ ]:
# TASK 5: Complete the connectivity and cycle checks using networkx.

def is_connected_undirected(G):
    return nx.___(G)   # fill in: which networkx function checks undirected connectivity?

def has_cycle_undirected(G):
    try:
        nx.find_cycle(G)
        return True
    except nx.exception.NetworkXNoCycle:
        return False

connected_example = nx.Graph([("A", "B"), ("A", "C"), ("B", "D")])
disconnected_example = nx.Graph([("A", "B"), ("C", "D")])
cycle_example = nx.Graph([("A", "B"), ("B", "C"), ("C", "A")])

assert is_connected_undirected(connected_example) == True
assert is_connected_undirected(disconnected_example) == False
assert has_cycle_undirected(cycle_example) == True
assert has_cycle_undirected(connected_example) == False
print("Section 5 checks passed.")


## Section 6 - Trees and DAGs

- A **tree** is a connected, acyclic *undirected* graph.
- A **DAG** is a directed graph with no directed cycles.


In [ ]:
# TASK 6: Complete these two one-line checks using networkx.

def is_tree(G):
    return nx.___(G)                     # fill in: networkx's tree-check function

def is_dag(G):
    return nx.___(G)                     # fill in: networkx's DAG-check function

tree_example = nx.Graph([("R", "A"), ("R", "B"), ("A", "C"), ("B", "D")])
not_a_tree = nx.Graph([("A", "B"), ("B", "C"), ("C", "A")])   # has a cycle

assert is_tree(tree_example) == True
assert is_tree(not_a_tree) == False
assert is_dag(student_bn) == True
print("Section 6 checks passed.")


## Section 7 - Topological Ordering: Kahn's Algorithm

Implement Kahn's algorithm **from scratch** (do not call `nx.topological_sort` inside this
function - we'll use it afterwards only to double-check your answer).

Recall the algorithm:
1. Compute in-degree of every node.
2. Put every node with in-degree 0 into a queue $Q$.
3. While $Q$ is not empty: pop a node $u$, append it to the ordering; for each child $v$ of
   $u$, decrement $v$'s in-degree, and if it hits 0, add $v$ to $Q$.
4. If every node was processed, the ordering is valid. Otherwise, the graph has a cycle.


In [ ]:
def kahns_algorithm(G):
    '''Returns a valid topological ordering (list of nodes) of DAG G,
    or raises a ValueError if G contains a cycle.'''
    in_degree = {node: G.in_degree(node) for node in G.nodes()}
    queue = [node for node in G.nodes() if in_degree[node] == ___]   # fill in

    ordering = []
    while queue:
        u = queue.pop(0)
        ordering.append(u)
        for v in G.successors(u):
            in_degree[v] -= ___          # fill in
            if in_degree[v] == ___:      # fill in
                queue.append(v)

    if len(ordering) != G.number_of_nodes():
        raise ValueError("Graph contains a directed cycle -- no valid topological ordering.")
    return ordering


order = kahns_algorithm(student_bn)
print("Topological order of Student BN:", order)

# A topological order must place every parent before every child.
for u, v in student_bn.edges():
    assert order.index(u) < order.index(v), f"{u} should come before {v}"
print("Section 7 checks passed: your ordering respects every edge.")


In [ ]:
# Cross-check against networkx's built-in implementation.
nx_order = list(nx.topological_sort(student_bn))
print("networkx's order:", nx_order)

for u, v in student_bn.edges():
    assert nx_order.index(u) < nx_order.index(v)
print("Both orderings are valid (they need not be identical -- ties can break either way).")


**Task 7b.** Run your `kahns_algorithm` on your personalized DAG (`my_dag`).


In [ ]:
my_order = kahns_algorithm(my_dag)
print("Topological order of your personalized DAG:", my_order)


## Section 8 - Skeleton and Polytrees

The **skeleton** of a DAG replaces every directed edge with an undirected one.
A DAG is a **polytree** (singly connected) if its skeleton has no undirected cycle.

In [ ]:
def get_skeleton(G):
    return G.___()   # fill in: which networkx method drops edge direction?

def is_polytree(G):
    skeleton = get_skeleton(G)
    return not has_cycle_undirected(skeleton)   # reuses your Section 5 function


assert is_polytree(student_bn) == True

# Add the extra edge D->I from lecture, which creates the undirected cycle D-G-I-D
not_polytree_bn = student_bn.copy()
not_polytree_bn.add_edge("D", "I")
assert is_polytree(not_polytree_bn) == False

print("Section 8 checks passed.")
print("Is your personalized DAG a polytree?", is_polytree(my_dag))


## Section 9 - Ancestors, Descendants, Non-Descendants

- $\text{Anc}(X)$: nodes with a directed path *to* $X$ — `nx.ancestors(G, X)`
- $\text{Desc}(X)$: nodes reachable *from* $X$ — `nx.descendants(G, X)`
- $\text{NonDesc}(X) = V \setminus (\{X\}\cup\text{Desc}(X))$

**Note (from our earlier discussion):** by this formula, $\text{Pa}(X)\subseteq\text{NonDesc}(X)$
always — the parents are non-descendants too. That is expected, not a bug: it's exactly what
lets the Local Markov Condition collapse "condition on everything non-descendant" down to
"condition on just the parents."


In [ ]:
def get_ancestors(G, node):
    return nx.___(G, node)     # fill in

def get_descendants(G, node):
    return nx.___(G, node)     # fill in

def get_non_descendants(G, node):
    V = set(G.nodes())
    desc = get_descendants(G, node)
    return V - {node} - desc   # V \ ({X} U Desc(X)) -- matches the lecture formula exactly

# Build and print the full table for the Student BN (matches the lecture's table)
print(f"{'Node':<6}{'Ancestors':<15}{'Descendants':<15}{'NonDescendants':<20}")
for node in ["D", "I", "G", "S", "L"]:
    anc = get_ancestors(student_bn, node)
    desc = get_descendants(student_bn, node)
    nondesc = get_non_descendants(student_bn, node)
    print(f"{node:<6}{str(anc):<15}{str(desc):<15}{str(nondesc):<20}")

assert get_non_descendants(student_bn, "G") == {"D", "I", "S"}
assert get_non_descendants(student_bn, "L") == {"D", "I", "G", "S"}
print("\nSection 9 checks passed -- note these match the FULL formula, parents included.")


In [ ]:
# TASK 9b: verify the Local Markov Condition holds as a SET relationship:
# Pa(X) is always a subset of NonDesc(X).

for node in student_bn.nodes():
    parents = set(student_bn.predecessors(node))
    nondesc = get_non_descendants(student_bn, node)
    assert parents.___(nondesc), f"Pa({node}) should be a subset of NonDesc({node})"   # fill in

print("Confirmed: Pa(X) is a subset of NonDesc(X) for every node, as expected.")


## Section 10 - Markov Blanket

$\text{MB}(X) = \text{Pa}(X)\cup\text{Ch}(X)\cup\text{CoParents}(X)$, where co-parents are the
*other* parents of $X$'s children.


In [ ]:
def markov_blanket(G, node):
    parents = set(G.predecessors(node))
    children = set(G.successors(node))

    co_parents = set()
    for child in children:
        co_parents |= set(G.predecessors(child))
    co_parents -= {node}   # a node is not its own co-parent

    return parents | children | ___   # fill in: which set is missing?

mb_G = markov_blanket(student_bn, "G")
print("MB(G) =", mb_G)

assert mb_G == {"D", "I", "L"}
print("Section 10 check passed -- matches the lecture's MB(G) = {D, I, L}.")


## Section 11 - V-Structures

A **v-structure** is a triple $(X, Z, Y)$ with $X\to Z \leftarrow Y$ and *no* edge between
$X$ and $Y$.


In [ ]:
def find_v_structures(G):
    v_structures = []
    for z in G.nodes():
        parents_of_z = list(G.predecessors(z))
        # check every pair of parents of z
        for x, y in itertools.combinations(parents_of_z, 2):
            if not G.has_edge(x, y) and not G.has_edge(___, ___):   # fill in: the reverse check
                v_structures.append((x, z, y))
    return v_structures

vs = find_v_structures(student_bn)
print("V-structures in Student BN:", vs)

assert ("D", "G", "I") in vs or ("I", "G", "D") in vs
print("Section 11 check passed.")


**Task 11b.** Find the v-structures in your personalized DAG.


In [ ]:
my_vs = find_v_structures(my_dag)
print("V-structures in your personalized DAG:", my_vs)


## Section 12 - Capstone: Putting It All Together

Using the functions you built above, run a **complete structural analysis** of your
personalized DAG (`my_dag`) in the cell below. This cell has no blanks - it's a summary
report using everything from Sections 2–11.


In [ ]:
print("="*60)
print(f"STRUCTURAL ANALYSIS REPORT -- seed = {SEED}")
print("="*60)

print("\nEdges:", list(my_dag.edges()))

print("\n--- Parents / Children ---")
for node in my_dag.nodes():
    pa, ch = get_parents_children(my_dag, node)
    print(f"{node}: Pa={pa}, Ch={ch}")

print("\n--- Topological order (Kahn's algorithm) ---")
print(kahns_algorithm(my_dag))

print("\n--- Skeleton / Polytree check ---")
print("Is polytree:", is_polytree(my_dag))

print("\n--- Ancestors / Descendants / Non-descendants ---")
print(f"{'Node':<6}{'Ancestors':<20}{'Descendants':<20}{'NonDescendants':<20}")
for node in my_dag.nodes():
    anc = get_ancestors(my_dag, node)
    desc = get_descendants(my_dag, node)
    nondesc = get_non_descendants(my_dag, node)
    print(f"{node:<6}{str(anc):<20}{str(desc):<20}{str(nondesc):<20}")

print("\n--- Markov Blankets ---")
for node in my_dag.nodes():
    print(f"MB({node}) = {markov_blanket(my_dag, node)}")

print("\n--- V-structures ---")
print(find_v_structures(my_dag))


### Reflection Questions (answer in this markdown cell)

1. **Is your personalized DAG a polytree?** Explain your answer using its skeleton. Does the
   skeleton contain an undirected cycle?

   _Your answer:_ ___

2. **Pick any node in your DAG with at least one v-structure at it.** Write out the Local
   Markov Condition for that node explicitly (i.e. $X \perp \text{NonDesc}(X) \mid \text{Pa}(X)$
   with the actual sets substituted in).

   _Your answer:_ ___

3. **Pick a node with an empty Markov blanket, if one exists (otherwise pick the node with the
   smallest Markov blanket).** What does an empty/small Markov blanket tell you about that
   node's position in the DAG?

   _Your answer:_ ___


---
## Submission Checklist
-  All `___` blanks replaced and every `assert` cell runs without error.
-  Section 12 capstone report printed successfully for your own `ROLL_NUMBER`.
-  Reflection questions answered in your own words.
-  Save the notebook: **File → Save** (or Ctrl+S).

**Rename the file as:** `Lab04_<YourRollNo>.ipynb` and convert to PDF before submitting.
